# 1. Overview

This notebook runs the checkpoint-only `tier` workflow as a thin operator console. It restores passed sample payloads and untiered passed manifests, runs the public `workflows.tier` workflow, reviews tier-stage typed lifecycle outputs, and publishes receipt-verified artifacts to Drive.

Workflow logic lives in `text_to_sign_production.workflows.tier`; this notebook only configures, reviews, executes, and summarizes the public workflow contract.

# 2. Operator Configuration

Set the repository revision, runtime roots, requested splits, and tier config paths for this run. These values are reviewed before runtime setup or workflow execution.

## 2.1 Repository and roots

Configure the repository checkout and runtime/Drive roots used by setup, restore, and publish operations.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "chore/core-layout-notebook-workflows"
PROJECT_ROOT = Path("/content/text-to-sign-production")
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/text-to-sign-production")

## 2.2 Tier workflow inputs

Configure the tier workflow inputs that are passed into the public workflow config.

In [ ]:
TIER_SPLITS = ("train", "val", "test")
FILTERS_CONFIG_RELATIVE_PATH = Path("configs/data/filters.yaml")
TIER_CONFIG_RELATIVE_PATH = Path("configs/data/tiers.yaml")

## 2.3 Configuration review

Review the selected values before preparing the runtime.

In [ ]:
print(f"Repository URL: {REPO_URL}")
print(f"Repository ref: {REPO_REF}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Drive project root: {DRIVE_PROJECT_ROOT}")
print(f"Tier splits: {TIER_SPLITS}")
print(f"Filters config: {FILTERS_CONFIG_RELATIVE_PATH}")
print(f"Tier config: {TIER_CONFIG_RELATIVE_PATH}")

# 3. Bootstrap Boundary

This isolated prelude prepares the Colab environment only: mount Drive, ensure system tooling, check out the repository, install dependencies, and make `src` importable. Skip this section when the environment is already prepared. Workflow restore, execution, validation, and publish steps start later in the runtime console.

## 3.1 Mount Drive

Mount Google Drive and confirm the configured Drive parent exists.

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
if not DRIVE_PROJECT_ROOT.parent.is_dir():
    raise FileNotFoundError(f"Drive MyDrive root is missing: {DRIVE_PROJECT_ROOT.parent}")
print(f"Drive mounted: {DRIVE_PROJECT_ROOT.parent}")

## 3.2 System packages

Ensure `zstd` is available for tar.zst restore and publish archive operations.

In [ ]:
import shutil

if shutil.which("zstd") is None:
    !sudo apt-get update
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to update apt package index.")

    !sudo apt-get install -y zstd
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError("Failed to install zstd.")
    print("Installed zstd.")
else:
    print("zstd is already available.")

!zstd --version
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("zstd command is not available after preflight.")

## 3.3 Repository checkout

Clone the requested repository revision into the configured runtime project root.

In [ ]:
%cd /content

if PROJECT_ROOT.exists():
    print(f"Removing stale repository checkout: {PROJECT_ROOT}")
    !rm -rf {PROJECT_ROOT}
    if globals().get("_exit_code", 1) != 0:
        raise RuntimeError(f"Failed to remove existing directory: {PROJECT_ROOT}")

!git clone {REPO_URL} {PROJECT_ROOT}
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to clone repository.")

!git -C {PROJECT_ROOT} checkout {REPO_REF}
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError(f"Failed to checkout revision {REPO_REF}.")

print(f"Repository ready: {PROJECT_ROOT}")
!git -C {PROJECT_ROOT} rev-parse HEAD
if globals().get("_exit_code", 1) != 0:
    raise RuntimeError("Failed to determine checked out revision.")

## 3.4 Install dependencies

Install the Colab dependency set from the checkout. Colab may restart the runtime after package installation; if that happens, global variables are cleared, so rerun Operator Configuration and Runtime Setup cells through this point before continuing.

In [ ]:
%cd {PROJECT_ROOT}
%pip install --upgrade pip
%pip install -r "requirements-colab.txt"
print("Repository dependencies installed from requirements-colab.txt.")

## 3.5 Add source tree to import path

Run this after dependency installation and after any runtime restart. It depends on restored globals such as `PROJECT_ROOT`, so rerun the configuration cells first if Colab restarted.


In [ ]:
import sys

%cd {PROJECT_ROOT}
source_path = PROJECT_ROOT / "src"
if str(source_path) not in sys.path:
    sys.path.append(str(source_path))
print(f"Repository src directory available on sys.path: {source_path}")

## 3.6 Workflow API import

Import the public tier workflow API from `text_to_sign_production.workflows.tier`.


In [ ]:
from text_to_sign_production.workflows.foundation.review import display_review_sections
from text_to_sign_production.workflows.tier import (
    TierWorkflow,
    TierWorkflowConfig,
)

# 4. Runtime Console: Plan

Build the tier workflow object and inspect the compact runtime restore plan before any restore command runs.

## 4.1 Build workflow config

Create the public tier workflow config from the reviewed operator inputs.

In [ ]:
tier_config = TierWorkflowConfig(
    project_root=PROJECT_ROOT,
    drive_project_root=DRIVE_PROJECT_ROOT,
    splits=TIER_SPLITS,
    filters_config_relpath=FILTERS_CONFIG_RELATIVE_PATH,
    tier_config_relpath=TIER_CONFIG_RELATIVE_PATH,
)

## 4.2 Instantiate workflow

Instantiate the workflow facade that owns runtime planning, restore, processing, reports, and publish calls.

In [ ]:
tier = TierWorkflow(tier_config)
print("Tier workflow instantiated.")

## 4.3 Plan runtime

Build the runtime restore plan without executing it.

In [ ]:
tier_runtime_plan = tier.plan_runtime()
print("Tier runtime plan built.")

## 4.4 Review runtime plan

Review a compact operator summary of the runtime plan.

In [ ]:
display_review_sections(tier.review_runtime_plan(tier_runtime_plan))

# 5. Runtime Console: Restore and Verify

Validate the plan, restore runtime inputs, and verify readiness as separate lifecycle transitions.

## 5.1 Validate runtime plan

Validate the planned runtime operations against the workflow layout before restore.

In [ ]:
tier.validate_runtime_plan(tier_runtime_plan)
print("Runtime plan validated.")

## 5.2 Execute runtime restore

Run the reviewed restore operations through the workflow facade.

In [ ]:
tier_restore_result = tier.restore_runtime(tier_runtime_plan)
print("Tier runtime restore complete.")

## 5.3 Review restore result

Review the compact restore execution result.

In [ ]:
display_review_sections(tier.review_runtime_restore(tier_restore_result))

## 5.4 Verify runtime

Run the explicit runtime readiness checks after restore.

In [ ]:
tier_runtime_verification = tier.verify_runtime(tier_runtime_plan)
print(f"Runtime {tier_runtime_verification.readiness_level.value} checked.")

## 5.5 Review runtime verification

Review the compact readiness outcome and any failed check examples.

In [ ]:
display_review_sections(tier.review_runtime_verification(tier_runtime_verification))

# 6. Runtime Console: Process

Run tier processing and review only the compact processing outcome by default.

## 6.1 Execute processing

Execute tier processing after runtime readiness has been checked.

In [ ]:
tier_bundle = tier.execute_processing(
    tier_runtime_plan.execution_inputs,
    tier_runtime_verification,
)
tier_result = tier_bundle.workflow_result
print("Tier workflow processing complete.")

## 6.2 Review compact processing summary

Review processing totals and membership totals without dumping every tier/sample row.

In [ ]:
display_review_sections(tier.review_processing(tier_bundle))

# 7. Runtime Console: Calibration, Outputs, and Reports

Review calibration, planned outputs, written reports, and final workflow outcome as separate compact states.

## 7.1 Review compact calibration summary

Review calibration surfaces and aggregate tier-report status without dumping every report row.

In [ ]:
display_review_sections(tier.review_calibration(tier_bundle))

## 7.2 Review planned outputs

Review planned output counts and key report paths.

In [ ]:
display_review_sections(tier.review_outputs(tier_result))

## 7.3 Write reports

Materialize tier report files; detailed rows are written to report artifacts instead of displayed by default.

In [ ]:
tier_report_artifacts = tier.write_reports(tier_bundle)
print(f"Tier reports written: {tier_report_artifacts.index_json_path}")

## 7.4 Review written report artifacts

Review compact report artifact counts and key paths.

In [ ]:
display_review_sections(tier.review_written_reports(tier_report_artifacts))

## 7.5 Review compact workflow summary

Review the compact workflow outcome before publishing.

In [ ]:
display_review_sections(
    tier.review_final_operator_summary(tier_bundle, tier_report_artifacts)
)

# 8. Runtime Console: Publish and Verify

Build, execute, verify, and review publish lifecycle states one step at a time.

## 8.1 Build publish plan

Build the publish plan from the processed bundle and written report artifacts.

In [ ]:
tier_publish_plan = tier.build_publish_plan(tier_bundle, tier_report_artifacts)
print("Tier publish plan built.")

## 8.2 Review compact publish plan

Review publish target and operation counts without dumping every target row.

In [ ]:
display_review_sections(tier.review_publish_plan(tier_publish_plan))

## 8.3 Execute publish

Run the publish plan through the workflow facade.

In [ ]:
tier_publish_execution = tier.execute_publish(tier_publish_plan)
print("Tier publish execution complete.")

## 8.4 Review compact publish execution

Review execution success/failure counts and failure examples only if present.

In [ ]:
display_review_sections(tier.review_publish_execution(tier_publish_execution))

## 8.5 Verify publish

Verify published targets after execution.

In [ ]:
tier_publish_verification = tier.verify_publish(tier_publish_plan)
print("Tier publish verification complete.")

## 8.6 Review compact publish verification

Review verification counts and failure examples without dumping every check row.

In [ ]:
display_review_sections(tier.review_publish_verification(tier_publish_verification))

## 8.7 Build publish result

Assemble the typed publish result after plan, execution, and verification are all available.

In [ ]:
tier_publish_result = tier.build_publish_result(
    tier_publish_plan,
    tier_publish_execution,
    tier_publish_verification,
)
print("Tier publish result built.")

## 8.8 Review compact publish result

Review the compact publish outcome for the operator console.

In [ ]:
display_review_sections(tier.review_publish_result(tier_publish_result))

# 9. Final Summary

Print one compact operator summary for workflow and publish outcomes. Detailed rows remain in report files and explicit detail review methods.

In [ ]:
display_review_sections(
    (
        *tier.review_final_operator_summary(tier_bundle, tier_report_artifacts),
        *tier.review_publish_result(tier_publish_result),
    )
)